# BTCUSDT 25-minute Price Predictor — Snowflake runbook

Runs the whole pipeline on a Snowflake **Container Runtime for ML** notebook with
4 × NVIDIA A10G (23 GB each), 48 vCPU, 100 GB RAM.

**Offline by design.** The only thing this notebook pulls from the network is Python
packages from PyPI. All market data arrives with the repository as compressed
`.btcz` shards — nothing calls Binance at train time.

| Step | Cell | Where it runs | Rough cost |
|---|---|---|---|
| 0 Environment check | 1–2 | — | seconds |
| 1 Materialise store | 3 | 48 vCPU | ~15–25 min, ~28 GB |
| 2 Train population | 4 | 4 × A10G | `--max-hours` |
| 3 Score + render video | 5–6 | 1 GPU + CPU | ~10 min |

Run the cells top to bottom.

In [ ]:
# --- 0. Locate the repo and install Python deps -----------------------------
# Point REPO at wherever you cloned/unpacked pridictior_snow8.
import os, sys, subprocess, pathlib, json

REPO = pathlib.Path(os.environ.get("BTCPRED_REPO", "/home/app/pridictior_snow8"))
if not (REPO / "src" / "btcpred").exists():
    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (cand / "src" / "btcpred").exists():
            REPO = cand
            break
assert (REPO / "src" / "btcpred").exists(), f"repo not found at {REPO}"
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)
print("repo:", REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", str(REPO / "requirements.txt")], check=False)
print("deps ok")

In [ ]:
# --- 1. Hardware + dataset sanity check -------------------------------------
import torch
from btcpred.data.dataset import ShardIndex
from btcpred.data.features import N_FEATURES, WARMUP

print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory/2**30:.1f} GiB")
print("cpus", os.cpu_count())

idx = ShardIndex(REPO / "data" / "shards")
print(idx)
print(f"features per second: {N_FEATURES} | analyser warm-up: {WARMUP/86400:.1f} days")

In [ ]:
# --- 2. Materialise the memmap store (CPU, parallel) ------------------------
# Decompresses every shard and precomputes the full Market Analyser matrix so
# the GPUs never wait on zstd or feature maths. Writes ~28 GB.
# Put STORE on fast local scratch, not on a stage.
STORE = os.environ.get("BTCPRED_STORE", "/tmp/btcstore")

cmd = [sys.executable, "scripts/materialize.py",
       "--shards", "data/shards", "--out", STORE,
       "--workers", str(min(16, os.cpu_count() or 8)),
       "--chunk", "4000000",           # 46 days/chunk: ~9% warm-up overhead
       "--feature-dtype", "float16",
       "--verify"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# --- 3. Train the population: one distinct model per GPU --------------------
# Not DDP. Rank 0 trains `base` (83 M), rank 1 `deep` (157 M), rank 2 `wide`
# (178 M), rank 3 `agile` (62 M), each with its own LR, path-basis size and
# stability weight. No epochs: cursors walk market time and every second is
# scored against the real tape 25 minutes later.
#
# Raise --max-hours for a serious run. Checkpoints land in runs/pop/<variant>/.
NGPU = max(1, torch.cuda.device_count())
MAX_HOURS = float(os.environ.get("BTCPRED_HOURS", 6))

cmd = ["torchrun", f"--nproc_per_node={NGPU}", "scripts/train.py",
       "--store", STORE, "--out", "runs/pop",
       "--batch", "24", "--stride", "1",
       "--max-hours", str(MAX_HOURS), "--amp", "bf16"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# --- 4. Training curves ------------------------------------------------------
import glob, json
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for f in sorted(glob.glob("runs/pop/*/metrics.jsonl")):
    rows = [json.loads(l) for l in open(f)]
    if not rows:
        continue
    name = rows[-1]["variant"]
    step = [r["step"] for r in rows]
    axes[0].plot(step, [r["nrmse"] for r in rows], label=name)
    axes[1].plot(step, [r["hit"] for r in rows], label=name)
    axes[2].plot(step, [r["reward"] for r in rows], label=name)
for ax, t in zip(axes, ["path nRMSE (lower=better)",
                        "directional hit rate", "reward"]):
    ax.set_title(t); ax.set_xlabel("market second (step)"); ax.grid(alpha=.3); ax.legend()
axes[1].axhline(0.5, ls=":", c="k")
plt.tight_layout(); plt.show()

In [ ]:
# --- 5. Replay a random day and render the 30 fps animation -----------------
# Every model predicts 25 minutes ahead, re-predicting as market time advances;
# accuracy bars are scored against the real stored tape.
cmd = [sys.executable, "scripts/render_video.py",
       "--store", STORE, "--ckpt-dir", "runs/pop",
       "--random-day", "--hours", "24",
       "--pred-stride", "20",      # predict every 20 market seconds
       "--frame-stride", "1",      # one frame per prediction -> 4320 frames
       "--fps", "30", "--out", "media/replay.mp4"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

from IPython.display import Video
Video("media/replay.mp4", embed=True, width=960)

## Tuning notes

* **Throughput.** Watch `market_s_per_s` in the training log. If it is low while
  GPU utilisation is also low, the host-side window gather is the bottleneck —
  raise `--batch` or run with the store on local NVMe.
* **Memory.** `wide` (178 M) at batch 24 and bf16 uses roughly 9 GB of the 23 GB
  card. Batch 48 is comfortable; batch 64 is about the ceiling.
* **Covering more history.** Each step advances every cursor by `--stride`
  seconds. `--stride 1` is true second-by-second; `--stride 5` covers five times
  more market time per step at the cost of correlated-sample efficiency.
* **Stability.** `LossWeights.stability` ties each second's forecast to the
  previous second's. Raise it if the dotted lines in the video jitter; the
  `stability_jump_sigma` column in the render summary measures exactly that.